## Práctica: Modelos probabilísticos de recuperación de información — BIM y BM25

###🎯 Objetivos

*   Comprender el principio de ranking probabilístico.
*   Implementar el modelo BIM (Binary Independence Model).
*   Extenderlo al modelo BM25, analizando sus ventajas.
*   Evaluar los resultados con un pequeño conjunto de documentos.


## 1. Preprocesado



### 1.1 Preparación del entorno

Vamos a preparar el entorno en el que haremos la práctica.

In [ ]:
# Instalación opcional (si no está disponible)
# !pip install pandas numpy

import numpy as np
import pandas as pd

Creamos un pequeño corpus de ejemplo:

In [ ]:
# Documentos de ejemplo
documents = {
    "D1": "information retrieval model",
    "D2": "probabilistic model for information retrieval",
    "D3": "vector space model and ranking",
    "D4": "bm25 model improves probabilistic retrieval",
}

query = ["probabilistic", "retrieval"]

### 1.2 Preprocesamiento básico

Por simplicidad realizaremos un breve preprocesado. Recordad que cuanto mejor sea esta etapa mejor será el resultado.

In [ ]:
def preprocess(doc):
    """Convierte texto a minúsculas, divide por espacios y elimina duplicados."""
    return list(set(doc.lower().split()))

# Creamos el vocabulario y los términos de cada documento
docs_terms = {d: preprocess(text) for d, text in documents.items()}
vocab = sorted({term for terms in docs_terms.values() for term in terms})
vocab


## 2. Principio de Ranking Probabilístico (PRP)

El **principio de ranking probabilístico** se basa en **ordenar los documentos** según la probabilidad de que sean **relevantes para una consulta** $Q$:

$$
P(R = 1 \mid D, Q)
$$

El objetivo es **estimar esta probabilidad** y **ordenar los documentos de forma decreciente** según ella.
	​


### 2.1 Supuestos del modelo BIM (*Binary Independence Model*)

El **BIM** es una **implementación práctica del PRP** bajo ciertas suposiciones simplificadoras:

- Representación **binaria** de los términos (presente / ausente).  
- **Independencia condicional** de los términos dada la relevancia.  
- No se requiere información de frecuencia ni de posición.

Cada documento $D$ recibe una puntuación (**score**) que estima su probabilidad de relevancia:

$$Score(D)=\sum_{i:d_i​=1} \log \frac{p_i​(1−s_i​)}{s_i​(1−p_i​)}=\sum_{i:d_i​=1} \log \frac{p_i}{1-p_i​} - \log \frac{s_i}{​1−s_i​}$$​
donde:

*  $p_i$ es $P(t∣R=1)$: probabilidad de que el término $t$ aparezca en documentos **relevantes**.  

*  $s_i$ es $P(t∣R=0)$: probabilidad de que el término $t$ aparezca en documentos **no relevantes**.

Por tanto la fórmula será la siguiente:
$$
Score(D) = \sum_{t \in Q}
\log \frac{P(t \mid R = 1)}{1 - P(t \mid R = 1)} -
\log \frac{P(t \mid R = 0)}{1 - P(t \mid R = 0)}
$$

Cuanto mayor sea el **score**, mayor es la probabilidad de que el documento sea relevante para la consulta.



### 2.2 Matriz de presencia binaria (para el modelo BIM)

Cada documento se representa por un vector binario (1 = término presente, 0 = ausente):

In [ ]:
# Construir matriz binaria de presencia
# crea un DataFrame con filas = documentos y columnas = términos
matrix = pd.DataFrame(0, index=documents.keys(), columns=vocab)

# completa el bucle para marcar con 1 los términos presentes
for d, terms in docs_terms.items():
    matrix.loc[d, terms] = 1

matrix

### 2.3. Implementación del modelo BIM

Para el modelo BIM, se estiman dos probabilidades:

*   $P(t∣R=1)$: probabilidad de aparición de un término en documentos relevantes.
*   $P(t∣R=0)$: probabilidad de aparición en documentos no relevantes.

Como no tenemos relevancia inicial, se asume **un modelo inicial con suavizado**, considerando el modelo clásico de Robertson y Sparck Jones (1976), es decir:

$$p_i=\frac{r_i + 0.5}{R + 1}$$

$$s_i=\frac{n_i - r_i + 0.5}{N - R + 1}$$

donde:

* $N$:	Número total de documentos.
* $n_i$: Número de documentos que contienen el término $t_i$.
* $R$:	Número total de documentos relevantes (si se conoce, o estimado).
* $r_i$: Número de documentos relevantes que contienen el término $t_i$.

**El suavizado en cada término permite evitar divisiones por cero, logaritmos de cero y que las probabilidades nunca sean exactamente 0 o 1.**

En tareas sin juicios de relevancia (por ejemplo, ranking inicial), una aproximación podría ser estimar simplemente:

$$p_i ≈ \frac{n_i + 0.5}{N+1} \quad \text{y} \quad s_i=1-p_i$$



### **Ejercicio:** Completar la impementación del código siguiente.

In [ ]:
def bim_score(query_terms, matrix, rel_docs=None):
    """
    Calcula el score BIM para cada documento.
    Si rel_docs no se pasa, se inicializa con un número aleatorio de documentos relevantes.
    """
    N = len(matrix)
    R = len(rel_docs) if rel_docs else 1
    scores = {}

    for d in matrix.index:
        score = 0
        for term in query_terms:
            n = matrix[term].sum()  # nº documentos con el término
            # Suavizado para evitar divisiones por cero
            p_t_R = ## completar
            p_t_NR = ## completar

            if matrix.loc[d, term] == 1:
                score += np.log(p_t_R / p_t_NR)
        scores[d] = score
    return scores


### **Preguntas:**
1. ¿Qué documentos se consideran más relevantes según el modelo?
2. ¿Qué limitaciones observas en el uso de representación binaria?
3. ¿Por qué crees que el modelo BIM puede no reflejar bien la frecuencia de los términos?

✏️ *Escribe tus respuestas debajo.*

### 2.4 Ranking según BIM

In [ ]:
scores_bim = bim_score(query, matrix)
ranking_bim = sorted(scores_bim.items(), key=lambda x: x[1], reverse=True)
pd.DataFrame(ranking_bim, columns=["Documento", "Score_BIM"])


### **Ejercicios:**

* Haz el mismo proceso que se ha realizado con los textos en inglés pero esta vez con los documentos en español que se encuentran más abajo.
* Una vez realizado, explica qué está ocurriendo y cuál es el motivo de obtener los resultados que se obtienen.

✏️ *Escribe tus respuestas debajo.*

In [ ]:
# Documentos de ejemplo
documentos = {
    "D1": "el modelo de recuperación de información",
    "D2": "el modelo de recuperación de información probabilístico",
    "D3": "el modelo espacio vectorial y ranking",
    "D4": "el modelo bm25 mejora la recuperación probabilística",
}

queryES = ["probabilístico", "probabilística", "recuperación"]

## 3. Extensión al modelo BM25

**BM25 (Best Matching 25)** es un modelo de ranking probabilístico que mejora del modelo BIM (*Binary Independence Model*), y se basa en el mismo principio probabilístico:

$$P(R=1∣D,Q)$$

es decir, la probabilidad de que un documento $D$ sea relevante dada una consulta $Q$.

El modelo BM25 mejora BIM introduciendo:

* Frecuencia de término ($t_f$): cuántas veces aparece un término en el documento.

* Longitud del documento: documentos más largos no deben ser penalizados ni favorecidos en exceso.

* Raro vs común: términos poco frecuentes en la colección son más discriminativos.

* Parámetros de ajuste $k_1$ y $b$

###3.1 Fórmula principal

$$
\text{BM25}(D, Q) = \sum_{t \in Q} \text{IDF}(t) \cdot
\frac{tf_D \cdot (k_1 + 1)}{tf_D + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}
$$

| Símbolo | Significado |
|----------|-------------|
| $tf_D$ | Frecuencia del término $t$ en el documento $D$. |
| $|D|$ | Longitud del documento (número de términos). |
| $\text{avgdl}$ | Longitud media de los documentos. |
| $k_1$ | Parámetro de control de saturación (1.2–2.0 típico). |
| $b$ | Factor de normalización por longitud (0.5–0.8 típico). |
| $\text{IDF}(t)$ | Inverse Document Frequency, mide la rareza del término. |

---


### 3.2 Cálculo del IDF

En el BIM, los valores de $p_i$ y $s_i$ para cada término se estima directamente de los datos de relevancia.

En BM25, se reemplazan por el **Inverse Document Frequency (IDF)**:
$$
\text{IDF}(t) = \log \frac{N - n_t + 0.5}{n_t + 0.5}
$$

donde:
- $N$: número total de documentos.  
- $n_t$: número de documentos que contienen el término $t$.

Cuanto más raro sea un término (menor $n_t$), mayor es su **peso** en el ranking.


### **Ejercicio:** Completar la impementación del código siguiente.

In [ ]:
def compute_idf(matrix):
    N = len(matrix)
    idf = {}
    for term in matrix.columns:
        n_t = #completar
        idf[term] = #completar
    return idf

idf = compute_idf(matrix)

### 3.3 Parámetros $k_1$ y $b$

| Parámetro | Rango típico | Efecto |
|------------|--------------|---------|
| $k_1$ | 1.2 – 2.0 | Controla la saturación: cuánto aumenta el peso si un término se repite. |
| $b$ | 0 – 1 | Controla cuánto afecta la longitud del documento. |

💡 Ejemplos:
- $b = 0$: la longitud del documento no influye.  
- $b = 1$: penaliza documentos largos proporcionalmente.



### **Ejercicio:** Completar la impementación del código siguiente.

In [ ]:
def bm25_score(query_terms, docs_terms, k1=1.2, b=1):
    scores = {}
    avg_len = np.mean([len(t) for t in docs_terms.values()])
    for d, terms in docs_terms.items():
        score = 0
        Ld = len(terms)
        for t in query_terms:
            tf = terms.count(t)
            if t in idf:
                score += #completar
        scores[d] = score
    return scores


### 3.4 Ranking con BM25

In [ ]:
scores_bm25 = bm25_score(query, docs_terms)
ranking_bm25 = sorted(scores_bm25.items(), key=lambda x: x[1], reverse=True)
pd.DataFrame(ranking_bm25, columns=["Documento", "Score_BM25"])


### **Ejercicios:**

- Prueba a cambiar los valores de los parámetros $k_1$ y $b$ y observa como cambia el ranking.

- Compara los resultados con los del modelo BIM.

## 4. Comparación final

In [ ]:
comparison = pd.DataFrame({
    "BIM": pd.Series(scores_bim),
    "BM25": pd.Series(scores_bm25)
}).fillna(0)

comparison.sort_values("BM25", ascending=False)


---

## Comparación con BIM

| Concepto | BIM | BM25 |
|-----------|-----|------|
| Fundamento teórico | Principio de ranking probabilístico | Mismo principio |
| Representación de término | Binaria | Frecuencia ponderada |
| Ajuste por longitud | ❌ No | ✅ Sí |
| Parámetros | ❌ No | ✅ \(k_1, b\) |
| Interpretación de pesos | Log-odds de relevancia | IDF + frecuencia normalizada |
| Eficiencia práctica | Limitada | Excelente, base de motores modernos |

---
En resumen:

> **BM25** es una **versión extendida y suavizada del BIM**,  
> que mantiene el espíritu probabilístico pero introduce:
> - Frecuencia de término real,  
> - Normalización de longitud,  
> - y calibración paramétrica para corpus grandes.

Por eso, BM25 se considera el **modelo estándar moderno de ranking probabilístico**, utilizado por motores como **Lucene**, **Elasticsearch** o **Whoosh**.

### **Ejercicios:**

1. ¿Qué documentos cambian de posición entre ambos modelos?

2. ¿Por qué BM25 tiende a mejorar los resultados respecto a BIM?

3. Ajusta los parámetros $k_1$ y $b$ y comenta cómo afecta al ranking.

4. Indicar de forma escrita qué resultados se obtendrían para estos documentos tanto para el caso BIM como BM25.

| Documento | Contenido | Longitud | Frecuencia de "información" |
|------------|------------|-----------|------------------------------|
| D1 | modelo de recuperación de información | 4 | 1 |
| D2 | recuperación de información, información y más información | 6 | 3 |


## Whoosh y el modelo de probabilidades


Para ordenar los resultados según relevancia, Whoosh implementa varios modelos de scoring, siendo uno de los más comunes el TF-IDF y el BM25.



### **Ejercicio:**

Adaptar el código proporcionado en la práctica 5 para que el score que utilice Whoosh sea el BM25.